# <font color="#2196F3">**PMC-Patients Clinical Data Preparation for GraphRAG**</font><br/>
### Normalization and Cut Generation for Knowledge Graph Distillation

This notebook transforms raw PMC-Patients case summaries into structured evaluation cuts:
**Output: `eval/data/pmc_cut.jsonl`** (`{patient_id, age, gender, text, condition_hint}`)

#### **Processing Steps**
1. **Age Parsing**: Transforms AST string representations (e.g. `[(60, 'year')]`) into standard strings (`60 years`).
2. **Gender Normalization**: Standardizes abbreviations (`M` -> `male`, `F` -> `female`).
3. **Clinical Text Filtering**: Ensures clinical notes fall within length boundaries (`300 <= len(text) <= 2500`).
4. **Condition Hint Detection**: Uses regex pattern matching to tag primary conditions (ARDS, COVID-19, pneumonia, carcinoma, stroke, etc.).
5. **Output Generation**: Writes formatted JSONL lines to `backend/eval/data/pmc_cut.jsonl` and `GTC25_DLI/data/pmc_cut.jsonl`.

## 1️⃣ Import Necessary Libraries & Setup Paths

In [1]:
import os
import sys
import ast
import csv
import json
import re
from pprint import pprint
import pandas as pd

# Allow large text fields
csv.field_size_limit(10**7)

# Workspace paths
NOTEBOOK_DIR = os.getcwd()
DATA_PREP_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in locals() else NOTEBOOK_DIR
WORKSPACE_ROOT = os.path.abspath(os.path.join(DATA_PREP_DIR, "..", "..", ".."))

CSV_SOURCE = os.path.join(WORKSPACE_ROOT, "backend", "eval", "data", "pmc_head.csv")
TARGET_CUT_JSONL = os.path.join(WORKSPACE_ROOT, "backend", "eval", "data", "pmc_cut.jsonl")
LOCAL_CUT_JSONL = os.path.join(WORKSPACE_ROOT, "GTC25_DLI", "data", "pmc_cut.jsonl")

print(f"Source CSV:    {CSV_SOURCE}")
print(f"Target Output: {TARGET_CUT_JSONL}")

Source CSV:    /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/backend/eval/data/pmc_head.csv
Target Output: /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/backend/eval/data/pmc_cut.jsonl


## 2️⃣ Define Clinical Parsing & Normalization Functions

In [2]:
# Regular expression for detecting clinical condition hints
CONDITION_PATTERN = re.compile(
    r"\b(ARDS|COVID-19|pneumonia|carcinoma|lymphoma|leukaemia|leukemia|sepsis|stroke|"
    r"myocardial infarction|tuberculosis|diabetes|hypertension|melanoma|sarcoma|"
    r"appendicitis|pancreatitis|hepatitis|nephropathy|anemia|anaemia|fracture)\b",
    re.I,
)

def parse_age(raw: str) -> str:
    """Parse raw AST age representation into human-readable format."""
    try:
        v = ast.literal_eval(raw)
        n, unit = v[0][0], v[0][1]
        return f"{int(n)} {unit}s" if n != 1 else f"1 {unit}"
    except Exception:
        return raw.strip()

def normalize_gender(raw: str) -> str:
    """Normalize gender abbreviation to lowercase full name."""
    g = raw.strip()
    return {"M": "male", "F": "female"}.get(g, g.lower())

## 3️⃣ Process Patients CSV into Structured JSONL Cut

In [3]:
def build_pmc_cut(csv_path, output_paths, target_count=100, min_chars=300, max_chars=2500):
    """Extract and format clinical cases into standardized JSONL format."""
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Source CSV missing: {csv_path}. Run 1_SEC_Data_Collection.ipynb first.")
        
    rows = []
    with open(csv_path, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        idx = {col.strip(): i for i, col in enumerate(header)}
        
        for row in reader:
            if len(row) != len(header):
                continue
            text = row[idx["patient"]].strip()
            if not (min_chars <= len(text) <= max_chars):
                continue
                
            m = CONDITION_PATTERN.search(text)
            patient_entry = {
                "patient_id": str(row[idx["patient_id"]]).strip(),
                "age": parse_age(row[idx["age"]]),
                "gender": normalize_gender(row[idx["gender"]]),
                "text": text,
                "condition_hint": (m.group(0).lower() if m else ""),
            }
            rows.append(patient_entry)
            if len(rows) >= target_count:
                break

    for p in output_paths:
        os.makedirs(os.path.dirname(p), exist_ok=True)
        with open(p, "w", encoding="utf-8") as out_f:
            for item in rows:
                out_f.write(json.dumps(item, ensure_ascii=False) + "\n")
        print(f"✅ Saved {len(rows)} records -> {p}")

    with_cond = sum(1 for r in rows if r["condition_hint"])
    print(f"\nSummary: {len(rows)} patients compiled ({with_cond} with detected condition hints)")
    return rows

patient_records = build_pmc_cut(
    CSV_SOURCE,
    [TARGET_CUT_JSONL, LOCAL_CUT_JSONL],
    target_count=100
)

✅ Saved 100 records -> /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/backend/eval/data/pmc_cut.jsonl
✅ Saved 100 records -> /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/pmc_cut.jsonl

Summary: 100 patients compiled (48 with detected condition hints)


## 4️⃣ Quality Validation and Exploratory Data Analysis (EDA)

In [4]:
# Inspect sample record
sample = patient_records[0]
print("--- Sample Record ---")
pprint(sample)

# Assert schema compliance
for r in patient_records:
    assert "patient_id" in r and r["patient_id"]
    assert "age" in r
    assert "gender" in r
    assert "text" in r and len(r["text"]) >= 300
    assert "condition_hint" in r
print("\n✅ Schema validation passed for all records!")

# Exploratory Analysis
df = pd.DataFrame(patient_records)
print("\n--- Gender Distribution ---")
print(df["gender"].value_counts())

print("\n--- Top Condition Hints ---")
print(df[df["condition_hint"] != ""]["condition_hint"].value_counts().head(10))

--- Sample Record ---
{'age': '60 years',
 'condition_hint': 'ards',
 'gender': 'male',
 'patient_id': '0',
 'text': 'This 60-year-old male was hospitalized due to moderate ARDS from '
         'COVID-19 with symptoms of fever, dry cough, and dyspnea. We '
         'encountered several difficulties during physical therapy on the '
         'acute ward. First, any change of position or deep breathing '
         'triggered coughing attacks that induced oxygen desaturation and '
         'dyspnea. To avoid rapid deterioration and respiratory failure, we '
         'instructed and performed position changes very slowly and '
         'step-by-step. In this way, a position change to the 135° prone '
         'position () took around 30 minutes. This approach was well tolerated '
         'and increased oxygen saturation, for example, on day 5 with 6 L/min '
         'of oxygen from 93% to 97%. Second, we had to adapt the breathing '
         'exercises to avoid prolonged coughing and oxygen